# Winter Storm Gianna at Pioneer CP10CNSM

Replay the six-day interval around February 1, 2026 at the OOI Pioneer Mid-Atlantic Bight Central Surface Mooring, off North Carolina. The four panels align buoy pressure, near-surface salinity, chlorophyll-a, and Imaging FlowCytobot regions of interest (ROIs) per mL. They show how the measurements changed through the event.

Run the following from the repository directory in an environment with Python 3.10+, pandas, and matplotlib. Choose an unused output directory for the CLI.

```sh
python3 ooi_replay.py --output-dir "$HOME/data/pioneer-gianna"
```

This notebook reads that directory by default. Set the Jupyter kernel's `PIONEER_REPLAY_DIR` environment variable to use another run. It writes `pioneer_storm.png` alongside the replay data.

[OOI’s Gianna event and image comparison](https://oceanobservatories.org/2026/09/ooi-plankton-imagery-and-co-located-environmental-data-now-ready-for-analysis/) · [CP10CNSM platform and instrument depths](https://oceanobservatories.org/site/CP10CNSM/)

In [ ]:
import json
import os
from datetime import timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
from IPython.display import Image, Markdown, display

replay_dir = Path(
    os.environ.get("PIONEER_REPLAY_DIR", "~/data/pioneer-gianna")
).expanduser()
if not (replay_dir / "replay.csv").is_file():
    raise FileNotFoundError(f"Run ooi_replay.py first. No replay.csv in {replay_dir}")

summary = json.loads((replay_dir / "summary.json").read_text(encoding="utf-8"))
hourly = pd.read_csv(replay_dir / "replay.csv")
hourly["hour_utc"] = pd.to_datetime(hourly["hour_utc"], utc=True)
hourly = hourly.set_index("hour_utc").sort_index()

start = pd.Timestamp(summary["start_inclusive_utc"])
end = pd.Timestamp(summary["end_exclusive_utc"])
storm_start = pd.Timestamp("2026-02-01T00:00:00Z")
storm_end = storm_start + pd.Timedelta(days=1)
image_times = pd.to_datetime(
    ["2026-02-01T00:25:46Z", "2026-02-01T15:25:45Z"], utc=True
)
variables = {
    "air_pressure": ("Air pressure (mbar)", "#345995"),
    "sea_water_practical_salinity": ("Practical salinity", "#177270"),
    "mass_concentration_of_chlorophyll_a_in_sea_water": (
        "Chlorophyll-a (µg/L)", "#49633D"
    ),
    "roipml": ("IFCB ROIs per mL", "#A75626"),
}

print(f"{summary['platform']} | {start} through {end} (exclusive) | {len(hourly)} hours")

## Hourly observations

The lines and points show medians of eligible observations in each UTC hour. Open circles identify hours that include data flagged **2 (not evaluated)**. Hours without eligible observations are blank. Separate IFCB points reflect its sparse sampling. The shaded band is February 1 UTC, and the vertical dotted lines mark the two published image samples.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9.5), sharex=True)

for ax, (variable, (label, color)) in zip(axes, variables.items()):
    values = hourly[f"{variable}_median"]
    has_qc2 = values.notna() & hourly[f"{variable}_qc2_n"].gt(0)
    ax.axvspan(storm_start, storm_end, color="#DCE8ED", alpha=0.75, zorder=0)
    for sample_time in image_times:
        ax.axvline(sample_time, color="#52606B", linestyle=":", linewidth=1)

    if variable == "roipml":
        passed = values.notna() & ~has_qc2
        ax.scatter(hourly.index[passed], values[passed], color=color, s=19)
    else:
        ax.plot(hourly.index, values, color=color, linewidth=1.4)

    ax.scatter(
        hourly.index[has_qc2], values[has_qc2], s=20,
        facecolors="white", edgecolors=color, linewidths=0.9, zorder=3
    )
    ax.set_ylabel(label)
    ax.grid(axis="y", linewidth=0.5, alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].set_xlim(start, end)
axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=8, tz=timezone.utc))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d  %H:%M", tz=timezone.utc))
axes[-1].set_xlabel("Time (UTC)")
fig.suptitle("Winter Storm Gianna · Pioneer Mid-Atlantic Bight", fontweight="bold")

legend = [
    Line2D([], [], marker="o", color="#52606B", markerfacecolor="white",
           linestyle="none", label="Includes QC flag 2"),
    Patch(facecolor="#DCE8ED", label="February 1 UTC"),
    Line2D([], [], color="#52606B", linestyle=":", label="IFCB image sample"),
]
fig.legend(handles=legend, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.045, 1, 0.955))
fig.savefig(replay_dir / "pioneer_storm.png", dpi=180, facecolor="white")
plt.show()

## Event metrics at source resolution

The pressure minimum is taken from the buoy observations within February 1 UTC. The IFCB and salinity table matches each published image timestamp to the nearest eligible observations within 15 minutes. The table reports their actual observation times, offsets, values, and flags. These are point comparisons at one mooring.

In [ ]:
def read_observations(source_key, products):
    source = summary["sources"][source_key]
    frame = pd.read_csv(replay_dir / source["observation_csv"])
    frame["time"] = pd.to_datetime(frame["time"], utc=True)
    for product in products:
        qc_ok = frame[f"{product}_qc_agg"].isin([1, 2])
        finite = frame[product].abs().lt(float("inf"))
        frame = frame[qc_ok & finite]
    return frame.sort_values("time")


pressure = read_observations("meteorology", ["air_pressure"])
storm_pressure = pressure[pressure["time"].between(storm_start, storm_end, inclusive="left")]
if storm_pressure.empty:
    print("No eligible buoy pressure observations on February 1 UTC in this replay.")
else:
    minimum = storm_pressure.loc[storm_pressure["air_pressure"].idxmin()]
    print(
        f"February 1 minimum observed pressure: {minimum['air_pressure']:.2f} mbar "
        f"at {minimum['time']:%H:%M} UTC (QC {minimum['air_pressure_qc_agg']})."
    )

pairs = pd.DataFrame({"Image UTC": image_times})
matched_sources = [
    ("plims", "IFCB", {
        "roipml": "ROIs/mL", "volana": "Volume (mL)", "roipml_qc_agg": "ROI/mL QC",
        "volana_qc_agg": "Volume QC"
    }, ["roipml", "volana"]),
    ("ctd", "CTD", {
        "sea_water_practical_salinity": "Practical salinity",
        "sea_water_practical_salinity_qc_agg": "Salinity QC"
    }, ["sea_water_practical_salinity"]),
]

for source, name, labels, products in matched_sources:
    time_column = f"{name} UTC"
    rows = read_observations(source, products)[["time", *labels]]
    rows = rows.rename(columns={"time": time_column, **labels})
    pairs = pd.merge_asof(
        pairs, rows, left_on="Image UTC", right_on=time_column,
        direction="nearest", tolerance=pd.Timedelta(minutes=15)
    )
    pairs[f"{name} offset (s)"] = (pairs[time_column] - pairs["Image UTC"]).dt.total_seconds()

table = pairs.copy()
numeric_columns = table.select_dtypes(include="number").columns
table[numeric_columns] = table[numeric_columns].round(3)
display(table)
roi = pairs["ROIs/mL"]
salinity = pairs["Practical salinity"]
if roi.notna().all() and roi.iloc[0] > 0:
    print(f"Later IFCB sample had {roi.iloc[1] / roi.iloc[0]:.2f}× the earlier ROIs/mL.")
if salinity.notna().all():
    print(f"Practical salinity changed by {salinity.iloc[1] - salinity.iloc[0]:+.3f}.")

## Optional representative IFCB imagery

The companion image sampler can place a contact sheet and its source manifest in `ifcb/` under the replay directory. The notebook also checks the replay directory itself. Set `PIONEER_IFCB_DIR` to use another image output directory. Each crop is an example ROI from a selected bin. These examples show appearance and do not estimate community composition.

In [ ]:
image_dir = Path(os.environ.get("PIONEER_IFCB_DIR", str(replay_dir / "ifcb"))).expanduser()
sheet_name = "pioneer-ifcb-contact-sheet.png"
if "PIONEER_IFCB_DIR" not in os.environ and not (image_dir / sheet_name).is_file():
    image_dir = replay_dir

sheet_path = image_dir / sheet_name
manifest_path = image_dir / "pioneer-ifcb-manifest.json"

if sheet_path.is_file():
    display(Image(filename=str(sheet_path)))
else:
    print("No local IFCB contact sheet found. See the published comparison linked above.")

if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for source in manifest["bins"]:
        display(Markdown(
            f"- {source['time_utc']} · [OOI sample {source['id']}]({source['source_url']})"
        ))

## Interpretation and attribution

ROIs per mL count image regions and can include particles and aggregates. Chlorophyll fluorescence is an optical proxy. These streams differ in cadence, support, and QC coverage. Aligning them in time describes co-variation at this mooring. An event replay alone cannot establish cause, species abundance, or forecast skill.

The nearby Central Shallow Water profiler CTD record published in Data Explorer ends in May 2025. With no profile observations in that record during Gianna, this replay uses near-surface and buoy measurements at the fixed Central Surface Mooring. [Profiler CTD coverage](https://erddap.dataexplorer.oceanobservatories.org/erddap/tabledap/ooi-cp12cnsw-wfp01-01-ctdpfp000.html)

Data are from the NSF Ocean Observatories Initiative, funded under NSF Cooperative Agreement No. 2244833 and managed by the Woods Hole Oceanographic Institution OOI Program Office. The replay summary records exact query URLs, units, time coverage and access time.

[Platform DOI](https://doi.org/10.58046/OOI-CP10CNSM) · [OOI quality control](https://oceanobservatories.org/quality-control/) · [OOI citation guidance](https://oceanobservatories.org/how-to-use-acknowledge-and-cite-data/)